# 05 — v2 Experiments: Augmentation, Fine-tuning, CV, Thresholds (Phase 6)

The Phase-4 baseline (notebook 03) answered *"does a frozen ResNet18 work at all?"* — yes, AUC 0.952 on one split. This notebook answers the harder, more honest questions, all driven by `src/experiments.py` and reproduced by `run_v2_experiments.py`.

## What was run

- **3 configs** on the same held-out test set: v1 frozen (no aug), v2 frozen+aug, v2 fine-tune `layer4`+aug.
- **Light augmentation** (horizontal flip, ±15° rotation, mild brightness/contrast jitter — no vertical flip, unnatural for retina).
- **Discriminative learning rates** for fine-tuning: new head lr=1e-3, unfrozen backbone lr=1e-4.
- **Early stopping** by best validation loss (fine-tuning overfits a 535-image set fast).
- **Bootstrap 95% CIs** (2,000 resamples) on the test set.
- **Screening threshold analysis** (highest threshold keeping sensitivity ≥ 0.95).
- **5-fold patient-level cross-validation** on the best config.

In [ ]:
import json
comp = json.load(open("../results/v2_comparison.json"))
for name in ["frozen_noaug_v1", "frozen_aug", "finetune_layer4_aug"]:
    r = comp[name]
    print(f"{name:22s} AUC={r['test_auc']:.3f} sens={r['test_sensitivity']:.3f} spec={r['test_specificity']:.3f}")

## Results

| Model | AUC | Sensitivity | Specificity |
|---|---|---|---|
| v1 frozen (baseline) | 0.952 | 0.892 | 0.971 |
| v2 frozen + aug | 0.965 | 0.938 | 0.882 |
| **v2 fine-tune layer4 + aug (best)** | **0.976** | **0.954** | **0.941** |

Best model 95% CIs: AUC [0.943, 0.998], sensitivity [0.90, 1.00], specificity [0.85, 1.00].

![model comparison](../figures/11_model_comparison.png)
![v2 ROC](../figures/12_roc_v2.png)

## Fine-tuning overfits — early stopping handles it

![v2 training curves](../figures/14_training_curves_v2.png)

Train loss → ~0 while val loss plateaus/drifts up: mild overfitting, expected on 535 images. The run keeps the best-val checkpoint (epoch 9), which is why `layer4` is only *partially* unfrozen rather than the whole backbone.

## Robustness: 5-fold patient-level CV

In [ ]:
cv = json.load(open("../results/cv_results.json"))
print("config:", cv["config"])
print("fold AUCs:", [round(a,3) for a in cv["fold_aucs"]])
print(f"mean AUC = {cv['mean_auc']:.3f} +/- {cv['std_auc']:.3f}")

**AUC 0.988 ± 0.008** across folds — the tight spread means the ~0.97–0.99 performance is stable across patient partitions, not a lucky split. (These folds share patients with the earlier held-out test set, so this is a stability check, not a second untouched test.)

![CV folds](../figures/15_cv_folds.png)

## Screening-oriented threshold

A screening tool should minimise *missed disease*, so the operating point matters more than the default 0.5 — lowering the threshold catches more disease at the cost of more false alarms. Trade-off on the test set (best model, source: `../results/threshold_sweep.json`):

| threshold | sensitivity | specificity | missed (FN) | false alarms (FP) |
|---|---|---|---|---|
| 0.30 | 0.969 | 0.824 | 2 | 6 |
| **0.40 (recommended)** | **0.969** | **0.912** | **2** | **3** |
| 0.50 (default) | 0.954 | 0.941 | 3 | 2 |
| 0.64 | 0.938 | 0.941 | 4 | 2 |

**Recommended: 0.40** — catches 63/65 disease (2 missed vs 3) for one extra false alarm. Going to 0.30 buys nothing (same sensitivity, double the false alarms). A real deployment threshold should be re-derived on a larger external set with clinical input.

## Error analysis: not explained by image quality

Only 5 test errors (3 missed glaucoma, 2 false alarms). Misclassified images have slightly lower mean quality (5.48 vs 6.04) but **not significantly** (Mann-Whitney U, p = 0.45, n=5). Errors aren't simply low-quality images — the human clinical reflection (notebook 04 TODO) still matters.

![error vs quality](../figures/10_error_vs_quality.png)